# KA SS 26 Challenge 4 - Notebook 4
## ConvNeXt-Tiny with log-mel, delta, and delta-delta channels

This is an independently evaluable single-model direction. The destination filename was retained exactly as requested, but the experiment is **ConvNeXt V1 Tiny**, not ConvNeXt V2.

Compliance choice: the primary checkpoint is `convnext_tiny.fb_in1k`, an explicitly supervised ImageNet-1K checkpoint. If and only if that checkpoint is unavailable, the notebook uses TorchVision's explicit `ConvNeXt_Tiny_Weights.IMAGENET1K_V1`. It never uses AudioSet, speech, Quran, Tajweed, CLIP/LAION, ImageNet-22K, or random initialization for training.

Frozen controls inherited from Notebook 2:

- exact `folds_v2` assignment embedded below with SHA-256 provenance;
- 16 kHz soxr decode, 8 s cache, 4 s deterministic evaluation crop;
- keep-first exact-hash representative for training only; all validation rows retained;
- no balanced sampling, CMVN, anomaly removal, exact-match patch, pseudo-labels, or TTA;
- official pooled and per-fold macro-F1 plus Notebook-2 sheikh stress metrics;
- best-epoch state is snapshotted and is the state used for test inference.

Shared augmentation contract: training uses +/-0.20 s center jitter, +/-6 dB gain, and at most two frequency masks (width 12) plus two time masks (width 28), each independently applied with probability 0.5. Validation and test are deterministic.

N4-only representation augmentation: with probability 0.30, raw log-mel dB is mildly warped along the **time axis only** at rate 0.96-1.04, then center-cropped or replicate-padded back to the original frame count. Delta and delta-delta are computed only afterward, keeping all channels mutually consistent.

The 8-second cache is anchored by a shared conservative active-center detector for clips longer than 4.25 seconds. Its exact thresholds and safety fallbacks are recorded in config and report; it never deletes a clip.

Expected Colab T4 runtime is roughly **18-30 minutes**, with normal variation from package download and I/O. Public leaderboard feedback is not used for selection.


In [1]:
# Pure-Python dependency bootstrap: no shell magics and no credential output.
import subprocess
import sys

REQUIRED_PACKAGES = [
    "kaggle",
    "timm==1.0.26",
    "soxr==1.0.0",
    "soundfile==0.13.1",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *REQUIRED_PACKAGES],
    check=True,
)
print("Dependencies ready.")


Dependencies ready.


In [2]:
import base64
import gc
import gzip
import hashlib
import io
import json
import math
import os
import random
import shutil
import stat
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import soxr
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from joblib import Parallel, delayed
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tqdm.auto import tqdm

T0 = time.time()
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("A CUDA GPU is required. Select a Colab T4 runtime before Run all.")

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything()
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

class CFG:
    COMPETITION = "ka-ss-26-challenge-4"
    SR = 16000
    CACHE_SEC = 8.0
    CROP_SEC = 4.0
    JITTER_SEC = 0.20
    GAIN_DB = 6.0
    N_FFT = 1024
    HOP = 160
    N_MELS = 128
    FMIN = 20
    FMAX = 8000
    DELTA_WIN = 9
    FREQ_MASK_MAX = 12
    TIME_MASK_MAX = 28
    TIME_WARP_PROB = 0.30
    TIME_WARP_RATE_MIN = 0.96
    TIME_WARP_RATE_MAX = 1.04
    ACTIVE_MIN_SEC = 4.25
    ACTIVE_FRAME_SEC = 0.040
    ACTIVE_HOP_SEC = 0.020
    ACTIVE_SMOOTH_FRAMES = 5
    ACTIVE_MIN_DYNAMIC_DB = 8.0
    ACTIVE_THRESHOLD_ABOVE_Q10_DB = 6.0
    ACTIVE_THRESHOLD_BELOW_Q90_DB = 24.0
    ACTIVE_SUPPORT_NEIGHBORHOOD = 5
    ACTIVE_SUPPORT_REQUIRED = 3
    ACTIVE_MIN_SUPPORTED_SEC = 0.60
    ACTIVE_MIN_SPAN_SEC = 1.00
    ACTIVE_MARGIN_SEC = 0.25
    ACTIVE_MAX_SHIFT_FRAC = 0.30
    PRIMARY_CHECKPOINT = "convnext_tiny.fb_in1k"
    FALLBACK_CHECKPOINT = "torchvision.ConvNeXt_Tiny_Weights.IMAGENET1K_V1"
    FOLDS = 5
    EPOCHS = 14
    BATCH = 16
    EVAL_BATCH = 32
    LR = 2e-4
    WD = 0.05
    LABEL_SMOOTH = 0.05
    DROPOUT = 0.30
    DROP_PATH = 0.10
    AMP = True
    NUM_WORKERS = 2
    EXPECTED_RUNTIME_MIN = "18-30"

CLASSES = ["Idgham", "Ikhfa", "Iqlab", "Izhar"]
CLS2IDX = {name: index for index, name in enumerate(CLASSES)}
NCLS = len(CLASSES)
CACHE_LEN = int(CFG.SR * CFG.CACHE_SEC)
CROP_LEN = int(CFG.SR * CFG.CROP_SEC)
OUT = Path("/content/notebook4_outputs")
OUT.mkdir(parents=True, exist_ok=True)
print("device:", torch.cuda.get_device_name(0), "| classes:", CLASSES)
print("expected runtime:", CFG.EXPECTED_RUNTIME_MIN, "minutes on one T4")


device: Tesla T4 | classes: ['Idgham', 'Ikhfa', 'Iqlab', 'Izhar']
expected runtime: 18-30 minutes on one T4


In [4]:
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

!mkdir -p /content/challenge4_data

!kaggle competitions download \
    -c ka-ss-26-challenge-4 \
    -p /content/challenge4_data \
    --force

100% 869M/869M [00:11<00:00, 76.5MB/s]



In [5]:
import zipfile, pathlib

p = pathlib.Path("/content/challenge4_data")
for z in p.glob("*.zip"):
    print("Extracting:", z.name)
    with zipfile.ZipFile(z) as f:
        f.extractall(p)

print("Done")

Extracting: ka-ss-26-challenge-4.zip
Done


In [6]:
# Secure Kaggle bootstrap. The JSON contents are copied but never printed or parsed here.
credential_source = Path("/content/kaggle.json")
if not credential_source.is_file():
    raise FileNotFoundError("Expected Kaggle credential file at /content/kaggle.json")

credential_dir = Path.home() / ".kaggle"
credential_dir.mkdir(parents=True, exist_ok=True)
os.chmod(credential_dir, 0o700)
credential_target = credential_dir / "kaggle.json"
shutil.copyfile(credential_source, credential_target)
os.chmod(credential_target, 0o600)
assert stat.S_IMODE(credential_target.stat().st_mode) == 0o600
print("Kaggle credentials configured with mode 600; credential values were not displayed.")

DATA_DIR = Path("/content/challenge4_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def is_competition_root(path):
    return (
        (path / "train.csv").is_file()
        and (path / "test.csv").is_file()
        and (path / "train").is_dir()
        and (path / "test").is_dir()
    )

def locate_competition_root(base):
    candidates = [base] + [p for p in base.rglob("*") if p.is_dir()]
    return next((p for p in candidates if is_competition_root(p)), None)

ROOT = locate_competition_root(DATA_DIR)
if ROOT is None:
    subprocess.run(
        [sys.executable, "-m", "kaggle", "competitions", "download",
         "-c", CFG.COMPETITION, "-p", str(DATA_DIR)],
        check=True,
    )
    archives = sorted(DATA_DIR.glob("*.zip"))
    if not archives:
        raise FileNotFoundError("Kaggle download completed but no ZIP archive was found")
    base_resolved = DATA_DIR.resolve()
    with zipfile.ZipFile(archives[-1]) as archive:
        for member in archive.infolist():
            target = (DATA_DIR / member.filename).resolve()
            if not target.is_relative_to(base_resolved):
                raise RuntimeError("Unsafe path in competition ZIP")
        archive.extractall(DATA_DIR)
    ROOT = locate_competition_root(DATA_DIR)

if ROOT is None:
    raise FileNotFoundError("Could not locate train.csv/test.csv and train/test WAV folders")
TRAIN_CSV, TEST_CSV = ROOT / "train.csv", ROOT / "test.csv"
TRAIN_DIR, TEST_DIR = ROOT / "train", ROOT / "test"
print("competition data root:", ROOT)


Kaggle credentials configured with mode 600; credential values were not displayed.
competition data root: /content/challenge4_data


In [7]:
# Frozen Notebook-2 fold payload. It is generated from the real synchronized folds_v2.csv.
FOLDS_SOURCE_SHA256 = "066B8033C8FD564BD6B90F357B5CE1A6F3212C93D3A10626A933D19C9EF25082"
FOLD_PAYLOAD_SHA256 = "06AD1FBA49231E2EE521E68437BBE3A00FAE5E2FD0B65FCF61B01DF2B6412EFF"
FOLD_PAYLOAD_GZIP_SHA256 = "E5541455B8CB1C7780B5838121A0925217B3FD83CA184CBA3AB9224AAED99D85"
EXPECTED_TRAIN_CSV_SHA256 = "EC948FC1D1D48C241E3EDF77A0EE592EAD548B9BABC707F27981F32AA1D50B13"
EXPECTED_TEST_CSV_SHA256 = "1276546D69FDD388C72D31A08D8B5E9CA86A6997F9D57232EA068B04163290E7"
FOLD_PAYLOAD_GZIP_B64 = (
    "..."
"AQA="
)

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest().upper()

def sha256_file(path):
    return sha256_bytes(Path(path).read_bytes())

compressed_payload = base64.b64decode(FOLD_PAYLOAD_GZIP_B64)
assert sha256_bytes(compressed_payload) == FOLD_PAYLOAD_GZIP_SHA256
payload_csv = gzip.decompress(compressed_payload)
assert sha256_bytes(payload_csv) == FOLD_PAYLOAD_SHA256
folds_payload = pd.read_csv(io.BytesIO(payload_csv))

expected_fold_columns = [
    "id", "fold_v2", "group_id_v2", "processed_hash", "is_extreme_anomaly"
]
assert list(folds_payload.columns) == expected_fold_columns
assert len(folds_payload) == 1515 and folds_payload.id.is_unique
folds_payload["is_extreme_anomaly"] = (
    folds_payload["is_extreme_anomaly"].astype(str).map({"True": True, "False": False})
)
assert folds_payload.is_extreme_anomaly.notna().all()

assert sha256_file(TRAIN_CSV) == EXPECTED_TRAIN_CSV_SHA256
assert sha256_file(TEST_CSV) == EXPECTED_TEST_CSV_SHA256
train_meta = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
SAMPLE_SUBMISSION_PATH = ROOT / "sample_submission.csv"
if SAMPLE_SUBMISSION_PATH.is_file():
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    print("optional sample_submission.csv found; schema will be verified")
else:
    sample_submission = pd.DataFrame({"id": test_df.id.to_numpy(dtype=int), "label_name": ""})
    print("no sample_submission.csv supplied; constructed exact id,label_name template from test.csv")
assert list(train_meta.columns) == ["id", "label_name", "sheikh_name"]
assert list(test_df.columns) == ["id", "sheikh_name"]
assert list(sample_submission.columns) == ["id", "label_name"]
assert sample_submission.id.tolist() == test_df.id.tolist()
assert train_meta.id.tolist() == folds_payload.id.tolist()
assert sorted(train_meta.label_name.unique().tolist()) == CLASSES

train_df = train_meta.copy()
for column in expected_fold_columns[1:]:
    train_df[column] = folds_payload[column].to_numpy()
train_df["y"] = train_df.label_name.map(CLS2IDX).astype(int)

assert train_df.fold_v2.value_counts().sort_index().to_dict() == {0: 303, 1: 303, 2: 304, 3: 302, 4: 303}
assert train_df.group_id_v2.nunique() == 1472
assert train_df.processed_hash.nunique() == 1472
assert int(train_df.is_extreme_anomaly.sum()) == 55
for fold in range(CFG.FOLDS):
    valid = train_df.fold_v2.eq(fold)
    assert not (set(train_df.loc[valid, "group_id_v2"]) & set(train_df.loc[~valid, "group_id_v2"]))
    assert not (set(train_df.loc[valid, "processed_hash"]) & set(train_df.loc[~valid, "processed_hash"]))

train_df["is_train_representative"] = ~train_df.duplicated("processed_hash", keep="first")
assert int(train_df.is_train_representative.sum()) == 1472
assert int((~train_df.is_train_representative).sum()) == 43
print("fold payload verified | source SHA-256:", FOLDS_SOURCE_SHA256)
print("fold counts:", train_df.fold_v2.value_counts().sort_index().to_dict())
print("training representatives:", int(train_df.is_train_representative.sum()), "/", len(train_df))


no sample_submission.csv supplied; constructed exact id,label_name template from test.csv
fold payload verified | source SHA-256: 066B8033C8FD564BD6B90F357B5CE1A6F3212C93D3A10626A933D19C9EF25082
fold counts: {0: 303, 1: 303, 2: 304, 3: 302, 4: 303}
training representatives: 1472 / 1515


In [8]:
# Exact N2 decode with a conservative shared active-center anchor for the 8-second cache.
def conservative_active_center(audio, sample_rate):
    duration = len(audio) / sample_rate
    raw_midpoint_sec = duration / 2.0
    base = {
        "center_shift_sec": 0.0,
        "active_trim_start_sec": 0.0,
        "active_trim_end_sec": duration,
        "active_center_used": False,
        "active_center_fallback": False,
        "active_center_fallback_reason": "short_clip",
    }
    if duration <= CFG.ACTIVE_MIN_SEC:
        return int(round(raw_midpoint_sec * sample_rate)), base
    base["active_center_fallback"] = True
    base["active_center_fallback_reason"] = "uncertain"
    if audio.size == 0 or not np.isfinite(audio).all():
        base["active_center_fallback_reason"] = "nonfinite"
        return int(round(raw_midpoint_sec * sample_rate)), base

    frame_length = int(round(CFG.ACTIVE_FRAME_SEC * sample_rate))
    hop_length = int(round(CFG.ACTIVE_HOP_SEC * sample_rate))
    if len(audio) < frame_length:
        base["active_center_fallback_reason"] = "too_short_for_rms"
        return int(round(raw_midpoint_sec * sample_rate)), base

    starts = np.arange(0, len(audio) - frame_length + 1, hop_length, dtype=np.int64)
    power = np.square(audio.astype(np.float64))
    cumulative = np.concatenate(([0.0], np.cumsum(power, dtype=np.float64)))
    frame_power = (cumulative[starts + frame_length] - cumulative[starts]) / frame_length
    rms_db = 10.0 * np.log10(np.maximum(frame_power, 1e-12))
    pad = CFG.ACTIVE_SMOOTH_FRAMES // 2
    smoothed_db = np.convolve(
        np.pad(rms_db, (pad, pad), mode="edge"),
        np.ones(CFG.ACTIVE_SMOOTH_FRAMES) / CFG.ACTIVE_SMOOTH_FRAMES,
        mode="valid",
    )
    if not np.isfinite(smoothed_db).all():
        base["active_center_fallback_reason"] = "nonfinite_rms"
        return int(round(raw_midpoint_sec * sample_rate)), base

    q10, q90 = np.quantile(smoothed_db, [0.10, 0.90])
    if q90 - q10 < CFG.ACTIVE_MIN_DYNAMIC_DB:
        base["active_center_fallback_reason"] = "low_dynamic_range"
        return int(round(raw_midpoint_sec * sample_rate)), base
    threshold_db = max(
        q10 + CFG.ACTIVE_THRESHOLD_ABOVE_Q10_DB,
        q90 - CFG.ACTIVE_THRESHOLD_BELOW_Q90_DB,
    )
    above = smoothed_db >= threshold_db
    support_count = np.convolve(
        above.astype(np.int16),
        np.ones(CFG.ACTIVE_SUPPORT_NEIGHBORHOOD, dtype=np.int16),
        mode="same",
    )
    supported = support_count >= CFG.ACTIVE_SUPPORT_REQUIRED
    supported_duration = float(supported.sum() * CFG.ACTIVE_HOP_SEC)
    if supported_duration < CFG.ACTIVE_MIN_SUPPORTED_SEC or not supported.any():
        base["active_center_fallback_reason"] = "insufficient_supported_activity"
        return int(round(raw_midpoint_sec * sample_rate)), base

    active_indices = np.flatnonzero(supported)
    activity_start_sec = starts[active_indices[0]] / sample_rate
    activity_end_sec = (starts[active_indices[-1]] + frame_length) / sample_rate
    if activity_end_sec - activity_start_sec < CFG.ACTIVE_MIN_SPAN_SEC:
        base["active_center_fallback_reason"] = "activity_span_too_short"
        return int(round(raw_midpoint_sec * sample_rate)), base

    trim_start_sec = max(0.0, activity_start_sec - CFG.ACTIVE_MARGIN_SEC)
    trim_end_sec = min(duration, activity_end_sec + CFG.ACTIVE_MARGIN_SEC)
    candidate_center_sec = (trim_start_sec + trim_end_sec) / 2.0
    center_shift_sec = candidate_center_sec - raw_midpoint_sec
    if abs(center_shift_sec) > CFG.ACTIVE_MAX_SHIFT_FRAC * duration:
        base["active_center_fallback_reason"] = "center_shift_too_large"
        return int(round(raw_midpoint_sec * sample_rate)), base

    diagnostics = {
        "center_shift_sec": float(center_shift_sec),
        "active_trim_start_sec": float(trim_start_sec),
        "active_trim_end_sec": float(trim_end_sec),
        "active_center_used": True,
        "active_center_fallback": False,
        "active_center_fallback_reason": "none",
    }
    return int(round(candidate_center_sec * sample_rate)), diagnostics

def process_one(path):
    audio, original_sr = sf.read(str(path), dtype="float32", always_2d=True)
    audio = audio.mean(axis=1)
    if int(original_sr) != CFG.SR:
        audio = soxr.resample(audio, int(original_sr), CFG.SR)
    audio = np.asarray(audio, dtype=np.float32)
    had_nonfinite = not np.isfinite(audio).all()
    audio = np.nan_to_num(audio)
    if len(audio) < 4096:
        audio = np.pad(audio, (0, 4096 - len(audio)))
    n_samples = len(audio)
    processed_hash = hashlib.md5(np.round(audio, 5).tobytes()).hexdigest()

    center_sample, diagnostics = conservative_active_center(audio, CFG.SR)
    if had_nonfinite:
        center_sample = n_samples // 2
        diagnostics.update({
            "center_shift_sec": 0.0,
            "active_trim_start_sec": 0.0,
            "active_trim_end_sec": n_samples / CFG.SR,
            "active_center_used": False,
            "active_center_fallback": n_samples / CFG.SR > CFG.ACTIVE_MIN_SEC,
            "active_center_fallback_reason": "nonfinite",
        })
    low = center_sample - CACHE_LEN // 2
    buffer = np.zeros(CACHE_LEN, dtype=np.float32)
    source_start = max(0, low)
    source_stop = min(n_samples, low + CACHE_LEN)
    target_start = source_start - low
    buffer[target_start:target_start + (source_stop - source_start)] = audio[source_start:source_stop]
    diagnostics["dur"] = n_samples / CFG.SR
    return buffer.astype(np.float16), processed_hash, diagnostics

def process_split(frame, folder, split_name):
    started = time.time()
    results = Parallel(n_jobs=4, backend="loky")(
        delayed(process_one)(folder / f"{int(audio_id)}.wav")
        for audio_id in tqdm(frame.id.to_numpy(), desc=f"decode {split_name}")
    )
    waves = np.stack([row[0] for row in results])
    hashes = np.asarray([row[1] for row in results])
    diagnostics = pd.DataFrame([row[2] for row in results])
    diagnostics.insert(0, "id", frame.id.to_numpy())
    print(split_name, "decoded in", round(time.time() - started, 1), "s | cache MB", round(waves.nbytes / 1e6))
    return waves, hashes, diagnostics

WTR, decoded_train_hashes, train_center_diag = process_split(train_df, TRAIN_DIR, "train")
WTE, decoded_test_hashes, test_center_diag = process_split(test_df, TEST_DIR, "test")
assert np.array_equal(decoded_train_hashes, train_df.processed_hash.to_numpy()), "Decoded hashes differ from frozen folds_v2"
assert train_df.id.tolist() == train_center_diag.id.tolist()
assert test_df.id.tolist() == test_center_diag.id.tolist()
diagnostic_columns = [
    "dur", "center_shift_sec", "active_trim_start_sec", "active_trim_end_sec",
    "active_center_used", "active_center_fallback", "active_center_fallback_reason",
]
for column in diagnostic_columns:
    train_df[column] = train_center_diag[column].to_numpy()
    test_df[column] = test_center_diag[column].to_numpy()
assert WTR.shape == (1515, CACHE_LEN) and WTE.shape == (747, CACHE_LEN)
assert np.isfinite(train_df.center_shift_sec).all() and np.isfinite(test_df.center_shift_sec).all()

def center_summary(frame):
    used = frame.active_center_used.to_numpy(dtype=bool)
    eligible = frame.dur.to_numpy() > CFG.ACTIVE_MIN_SEC
    left_trim = frame.active_trim_start_sec.to_numpy()
    right_trim = frame.dur.to_numpy() - frame.active_trim_end_sec.to_numpy()
    return {
        "n": int(len(frame)),
        "eligible_over_4p25s": int(eligible.sum()),
        "active_center_used": int(used.sum()),
        "fallback_count_eligible": int(frame.active_center_fallback.sum()),
        "changed_at_least_0p25s_count": int((frame.center_shift_sec.abs() >= 0.25).sum()),
        "changed_at_least_0p25s_percent": float(100 * (frame.center_shift_sec.abs() >= 0.25).mean()),
        "mean_left_trim_sec_when_used": float(left_trim[used].mean()) if used.any() else 0.0,
        "mean_right_trim_sec_when_used": float(right_trim[used].mean()) if used.any() else 0.0,
    }

CENTER_DIAGNOSTICS = {
    "train": center_summary(train_df),
    "test": center_summary(test_df),
}
print("decoded hash alignment with frozen folds: PASSED")
print("active-center diagnostics:", CENTER_DIAGNOSTICS)


decode train:   0%|          | 0/1515 [00:00<?, ?it/s]

train decoded in 10.4 s | cache MB 388


decode test:   0%|          | 0/747 [00:00<?, ?it/s]

test decoded in 3.7 s | cache MB 191
decoded hash alignment with frozen folds: PASSED
active-center diagnostics: {'train': {'n': 1515, 'eligible_over_4p25s': 246, 'active_center_used': 221, 'fallback_count_eligible': 25, 'changed_at_least_0p25s_count': 29, 'changed_at_least_0p25s_percent': 1.914191419141914, 'mean_left_trim_sec_when_used': 0.13520361990950228, 'mean_right_trim_sec_when_used': 0.12284530542986424}, 'test': {'n': 747, 'eligible_over_4p25s': 116, 'active_center_used': 108, 'fallback_count_eligible': 8, 'changed_at_least_0p25s_count': 21, 'changed_at_least_0p25s_percent': 2.8112449799196786, 'mean_left_trim_sec_when_used': 0.11731481481481482, 'mean_right_trim_sec_when_used': 0.1397158564814815}}


## Representation and model

Every waveform uses the same deterministic center region before feature extraction. For clips longer than 4.25 seconds, a conservative 40 ms RMS / 20 ms hop activity detector (5-frame smoothing and 3-of-5 support) may move the 8-second cache anchor; uncertain cases and shifts beyond 30% of clip duration fall back to the raw midpoint. Short clips use raw-midpoint symmetric padding. No clip is removed.

Each 4-second crop becomes a native rectangular `3 x 128 x 401` tensor. Training augmentation order is exact:

1. waveform center jitter (+/-0.20 s) and gain (+/-6 dB);
2. raw power log-mel dB;
3. N4-only pitch-preserving time-axis warp at rate 0.96-1.04 with probability 0.30, center-cropped or replicate-padded to 401 frames;
4. first temporal delta (`win_length=9`, replicate padding), then delta-delta;
5. independent global standardization of each channel;
6. shared-channel SpecAugment: up to two frequency masks (max 12) and two time masks (max 28), each with probability 0.5.

Validation and test use the fixed active-center region and raw log-mel -> deltas -> standardization only: no jitter, gain, time warp, or SpecAugment. No resizing or square distortion is applied.


In [9]:
class DeltaMelFront(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.SR,
            n_fft=CFG.N_FFT,
            hop_length=CFG.HOP,
            n_mels=CFG.N_MELS,
            f_min=CFG.FMIN,
            f_max=CFG.FMAX,
            power=2.0,
        )
        self.db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)

    @staticmethod
    def _warp_one(logmel_sample, rate):
        original_t = logmel_sample.shape[-1]
        warped_t = max(1, int(round(original_t * rate)))
        warped = F.interpolate(
            logmel_sample[None, None],
            size=(logmel_sample.shape[-2], warped_t),
            mode="bilinear",
            align_corners=False,
        )
        if warped_t > original_t:
            start = (warped_t - original_t) // 2
            warped = warped[..., start:start + original_t]
        elif warped_t < original_t:
            missing = original_t - warped_t
            left = missing // 2
            right = missing - left
            warped = F.pad(warped, (left, right, 0, 0), mode="replicate")
        result = warped[0, 0]
        assert result.shape == logmel_sample.shape
        return result

    def _time_warp(self, raw_logmel_db, probability):
        warped_batch = []
        for sample in raw_logmel_db:
            if random.random() < probability:
                rate = random.uniform(CFG.TIME_WARP_RATE_MIN, CFG.TIME_WARP_RATE_MAX)
                sample = self._warp_one(sample, rate)
            warped_batch.append(sample)
        result = torch.stack(warped_batch)
        assert result.shape == raw_logmel_db.shape
        return result

    def forward(self, waveform, apply_time_warp=False, warp_probability=None):
        raw_logmel_db = self.db(self.mel(waveform))
        if apply_time_warp:
            probability = CFG.TIME_WARP_PROB if warp_probability is None else warp_probability
            raw_logmel_db = self._time_warp(raw_logmel_db, probability)
        # Deltas are deliberately computed after the optional raw-dB time warp.
        delta = torchaudio.functional.compute_deltas(
            raw_logmel_db, win_length=CFG.DELTA_WIN, mode="replicate"
        )
        delta2 = torchaudio.functional.compute_deltas(
            delta, win_length=CFG.DELTA_WIN, mode="replicate"
        )
        features = torch.stack([raw_logmel_db, delta, delta2], dim=1)
        mean = features.mean(dim=(-2, -1), keepdim=True)
        std = features.std(dim=(-2, -1), keepdim=True)
        return (features - mean) / (std + 1e-6)

def _shared_mask(tensor, axis, maximum_width):
    width = random.randint(1, maximum_width)
    length = tensor.shape[axis]
    width = min(width, length)
    start = random.randint(0, length - width)
    slices = [slice(None)] * tensor.ndim
    slices[axis] = slice(start, start + width)
    tensor[tuple(slices)] = 0

def shared_spec_augment(features, probability=0.5):
    augmented = features.clone()
    for axis, width in ((2, CFG.FREQ_MASK_MAX), (2, CFG.FREQ_MASK_MAX),
                        (3, CFG.TIME_MASK_MAX), (3, CFG.TIME_MASK_MAX)):
        if random.random() < probability:
            _shared_mask(augmented, axis, width)
    return augmented

class WaveDataset(torch.utils.data.Dataset):
    def __init__(self, waves, indices, labels=None, train=False):
        self.waves = waves
        self.indices = np.asarray(indices)
        self.labels = labels
        self.train = train
        self.jitter = int(CFG.JITTER_SEC * CFG.SR) if train else 0
        self.center_start = CACHE_LEN // 2 - CROP_LEN // 2

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        row = int(self.indices[item])
        offset = random.randint(-self.jitter, self.jitter) if self.jitter else 0
        start = int(np.clip(self.center_start + offset, 0, CACHE_LEN - CROP_LEN))
        waveform = self.waves[row, start:start + CROP_LEN].astype(np.float32)
        if self.train:
            waveform *= np.float32(10 ** (random.uniform(-CFG.GAIN_DB, CFG.GAIN_DB) / 20))
        waveform = torch.from_numpy(np.ascontiguousarray(waveform))
        if self.labels is None:
            return waveform
        return waveform, torch.tensor(int(self.labels[row]), dtype=torch.long)

def build_untrained_primary():
    import timm
    return timm.create_model(
        CFG.PRIMARY_CHECKPOINT,
        pretrained=False,
        num_classes=NCLS,
        in_chans=3,
        drop_rate=CFG.DROPOUT,
        drop_path_rate=CFG.DROP_PATH,
    )

def build_pretrained_model(source):
    if source == CFG.PRIMARY_CHECKPOINT:
        import timm
        return timm.create_model(
            CFG.PRIMARY_CHECKPOINT,
            pretrained=True,
            num_classes=NCLS,
            in_chans=3,
            drop_rate=CFG.DROPOUT,
            drop_path_rate=CFG.DROP_PATH,
        )
    if source == CFG.FALLBACK_CHECKPOINT:
        from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny
        model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Sequential(
            nn.Dropout(CFG.DROPOUT), nn.Linear(in_features, NCLS)
        )
        return model
    raise ValueError(f"Unapproved checkpoint source: {source}")

def resolve_pretrained_source():
    try:
        probe = build_pretrained_model(CFG.PRIMARY_CHECKPOINT)
        del probe
        gc.collect()
        torch.cuda.empty_cache()
        return CFG.PRIMARY_CHECKPOINT
    except Exception as primary_error:
        print("Primary supervised ImageNet-1K checkpoint unavailable:", type(primary_error).__name__)
        try:
            probe = build_pretrained_model(CFG.FALLBACK_CHECKPOINT)
            del probe
            gc.collect()
            torch.cuda.empty_cache()
            return CFG.FALLBACK_CHECKPOINT
        except Exception as fallback_error:
            raise RuntimeError(
                "Neither approved generic ImageNet-1K ConvNeXt-Tiny checkpoint is available; "
                "refusing to train from random or audio-pretrained weights."
            ) from fallback_error

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)


In [10]:
# Synthetic preflight only: no data labels, no pretrained download, no optimization.
seed_everything()
preflight_front = DeltaMelFront().to(DEVICE).eval()
synthetic_wave = torch.randn(2, CROP_LEN, device=DEVICE)
with torch.inference_mode():
    deterministic_features_a = preflight_front(synthetic_wave, apply_time_warp=False)
    deterministic_features_b = preflight_front(synthetic_wave, apply_time_warp=False)
    random.seed(123)
    warped_features = preflight_front(
        synthetic_wave, apply_time_warp=True, warp_probability=1.0
    )
assert deterministic_features_a.shape == (2, 3, 128, 401)
assert warped_features.shape == deterministic_features_a.shape
assert deterministic_features_a.numel() > 0 and warped_features.numel() > 0
assert torch.isfinite(deterministic_features_a).all().item()
assert torch.isfinite(warped_features).all().item()
assert torch.equal(deterministic_features_a, deterministic_features_b), "Validation/test front must be deterministic"
channel_means = deterministic_features_a.mean(dim=(-2, -1))
channel_stds = deterministic_features_a.std(dim=(-2, -1))
assert torch.allclose(channel_means, torch.zeros_like(channel_means), atol=2e-4)
assert torch.allclose(channel_stds, torch.ones_like(channel_stds), atol=2e-4)

random.seed(123)
mask_probe = shared_spec_augment(torch.ones(2, 3, 128, 401), probability=1.0)
zero_map = mask_probe.eq(0)
assert torch.equal(zero_map[:, 0], zero_map[:, 1])
assert torch.equal(zero_map[:, 1], zero_map[:, 2])

preflight_model = build_untrained_primary().to(DEVICE, memory_format=torch.channels_last).eval()
with torch.inference_mode():
    synthetic_logits = preflight_model(
        deterministic_features_a.contiguous(memory_format=torch.channels_last)
    )
assert synthetic_logits.shape == (2, 4)
print("synthetic preflight PASSED:", tuple(deterministic_features_a.shape), "->", tuple(synthetic_logits.shape))
print("time warp fixed-shape/finite and deterministic no-warp assertions: PASSED")
del preflight_model, preflight_front, synthetic_wave, deterministic_features_a
del deterministic_features_b, warped_features, synthetic_logits, mask_probe
gc.collect()
torch.cuda.empty_cache()
seed_everything()


synthetic preflight PASSED: (2, 3, 128, 401) -> (2, 4)
time warp fixed-shape/finite and deterministic no-warp assertions: PASSED


In [11]:
SHEIKHS = sorted(set(train_df.sheikh_name) | set(test_df.sheikh_name))
sheikh_counts = train_df.sheikh_name.value_counts()
SMALL_SHEIKHS = sheikh_counts[sheikh_counts < 100].index.tolist()

def full_eval(y_true, y_pred, frame):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    output = {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "accuracy": float((y_true == y_pred).mean()),
        "per_class": {
            name: float(value)
            for name, value in zip(
                CLASSES,
                f1_score(y_true, y_pred, average=None, labels=range(NCLS)),
            )
        },
    }
    sheikh_rows = []
    for sheikh in SHEIKHS:
        mask = frame.sheikh_name.to_numpy() == sheikh
        if not mask.any():
            continue
        present = sorted(set(y_true[mask].tolist()))
        sheikh_rows.append({
            "sheikh": sheikh,
            "n": int(mask.sum()),
            "n_classes_present": len(present),
            "acc": float((y_true[mask] == y_pred[mask]).mean()),
            "present_macro_f1": float(f1_score(
                y_true[mask], y_pred[mask], average="macro",
                labels=present, zero_division=0,
            )),
        })
    per_sheikh = pd.DataFrame(sheikh_rows)
    output["per_sheikh"] = per_sheikh
    output["mean_sheikh_acc"] = float(per_sheikh.acc.mean())
    output["mean_present_macro_f1"] = float(per_sheikh.present_macro_f1.mean())

    def subset_score(mask):
        present = sorted(set(y_true[mask].tolist()))
        return float(f1_score(
            y_true[mask], y_pred[mask], average="macro",
            labels=present, zero_division=0,
        ))

    small_mask = frame.sheikh_name.isin(SMALL_SHEIKHS).to_numpy()
    marsh_mask = frame.sheikh_name.to_numpy() == "abdulah_marsh"
    output["small_sheikh_macro_f1"] = subset_score(small_mask)
    output["marsh_macro_f1"] = subset_score(marsh_mask)
    output["non_marsh_macro_f1"] = subset_score(~marsh_mask)
    return output

def print_eval(metrics, title):
    print("---", title, "---")
    print("official macro-F1:", round(metrics["macro_f1"], 4), "| accuracy:", round(metrics["accuracy"], 4))
    print("per-class F1:", {k: round(v, 4) for k, v in metrics["per_class"].items()})
    print("small-sheikh:", round(metrics["small_sheikh_macro_f1"], 4),
          "| non-marsh:", round(metrics["non_marsh_macro_f1"], 4),
          "| marsh:", round(metrics["marsh_macro_f1"], 4))

def make_loader(dataset, batch_size, shuffle=False):
    generator = torch.Generator()
    generator.manual_seed(SEED)
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True,
        persistent_workers=CFG.NUM_WORKERS > 0,
        worker_init_fn=seed_worker,
        generator=generator,
    )

def run_fold(train_indices, valid_indices, fold, model_source):
    seed_everything()
    train_indices = np.asarray(train_indices)
    train_indices = train_indices[
        train_df.is_train_representative.to_numpy()[train_indices]
    ]
    labels = train_df.y.to_numpy()
    train_loader = make_loader(
        WaveDataset(WTR, train_indices, labels=labels, train=True),
        CFG.BATCH,
        shuffle=True,
    )
    valid_loader = make_loader(
        WaveDataset(WTR, valid_indices, labels=labels, train=False),
        CFG.EVAL_BATCH,
        shuffle=False,
    )

    front = DeltaMelFront().to(DEVICE).eval()
    model = build_pretrained_model(model_source).to(
        DEVICE, memory_format=torch.channels_last
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WD)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=CFG.LR,
        total_steps=max(1, len(train_loader)) * CFG.EPOCHS,
        pct_start=0.15,
        div_factor=10,
        final_div_factor=50,
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTH)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.AMP)

    best_f1 = -1.0
    best_epoch = -1
    best_probabilities = None
    best_state = None
    started = time.time()

    for epoch in range(1, CFG.EPOCHS + 1):
        model.train()
        for waveforms, targets in train_loader:
            waveforms = waveforms.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            with torch.no_grad():
                features = shared_spec_augment(front(waveforms, apply_time_warp=True))
                features = features.contiguous(memory_format=torch.channels_last)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=CFG.AMP):
                loss = criterion(model(features), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        model.eval()
        epoch_probabilities = []
        with torch.inference_mode():
            for waveforms, _ in valid_loader:
                waveforms = waveforms.to(DEVICE, non_blocking=True)
                features = front(waveforms, apply_time_warp=False).contiguous(memory_format=torch.channels_last)
                with torch.cuda.amp.autocast(enabled=CFG.AMP):
                    logits = model(features)
                epoch_probabilities.append(logits.float().softmax(1).cpu().numpy())
        epoch_probabilities = np.concatenate(epoch_probabilities)
        epoch_f1 = f1_score(
            labels[valid_indices], epoch_probabilities.argmax(1), average="macro"
        )
        if epoch_f1 > best_f1:
            best_f1 = float(epoch_f1)
            best_epoch = epoch
            best_probabilities = epoch_probabilities.copy()
            # Critical fix over Notebook 2: snapshot the state at the same best epoch.
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
        print(f"fold {fold} epoch {epoch:02d}/{CFG.EPOCHS}: macro-F1 {epoch_f1:.4f}")

    assert best_state is not None and best_probabilities is not None
    elapsed = time.time() - started
    del model, optimizer, scheduler, scaler, front, train_loader, valid_loader
    gc.collect()
    torch.cuda.empty_cache()
    return {
        "fold": fold,
        "macro_f1": best_f1,
        "best_epoch": best_epoch,
        "n_train": int(len(train_indices)),
        "seconds": float(elapsed),
        "probabilities": best_probabilities,
        "state": best_state,
    }


In [12]:
# Five-fold training. This is the only intentionally expensive cell.
MODEL_SOURCE = resolve_pretrained_source()
print("resolved approved checkpoint:", MODEL_SOURCE)
print("No audio/speech/domain-specific pretrained weights are used.")

labels = train_df.y.to_numpy()
OOF = np.zeros((len(train_df), NCLS), dtype=np.float32)
fold_results = []
best_states = []
training_started = time.time()

for fold in range(CFG.FOLDS):
    valid_indices = np.flatnonzero(train_df.fold_v2.to_numpy() == fold)
    train_indices = np.flatnonzero(train_df.fold_v2.to_numpy() != fold)
    result = run_fold(train_indices, valid_indices, fold, MODEL_SOURCE)
    OOF[valid_indices] = result.pop("probabilities")
    best_states.append(result.pop("state"))
    fold_results.append(result)

TRAIN_SECONDS = time.time() - training_started
assert np.isfinite(OOF).all()
assert np.allclose(OOF.sum(axis=1), 1.0, atol=2e-5)
oof_predictions = OOF.argmax(axis=1)
FINAL_EVAL = full_eval(labels, oof_predictions, train_df)
fold_metrics = pd.DataFrame(fold_results).sort_values("fold").reset_index(drop=True)
recomputed_fold_f1 = []
for fold in range(CFG.FOLDS):
    mask = train_df.fold_v2.to_numpy() == fold
    recomputed_fold_f1.append(float(f1_score(labels[mask], oof_predictions[mask], average="macro")))
assert np.allclose(fold_metrics.macro_f1.to_numpy(), recomputed_fold_f1, atol=1e-12)

print_eval(FINAL_EVAL, "Notebook 4 pooled OOF")
print("per-fold macro-F1:", [round(value, 4) for value in recomputed_fold_f1])
print("training minutes:", round(TRAIN_SECONDS / 60, 1))


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

resolved approved checkpoint: convnext_tiny.fb_in1k
No audio/speech/domain-specific pretrained weights are used.


/tmp/ipykernel_1250/2997398431.py:108: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=CFG.AMP)
/tmp/ipykernel_1250/2997398431.py:125: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CFG.AMP):
/tmp/ipykernel_1250/2997398431.py:138: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CFG.AMP):


fold 0 epoch 01/14: macro-F1 0.5341


...

fold 4 epoch 12/14: macro-F1 0.9573


/tmp/ipykernel_1250/2997398431.py:125: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CFG.AMP):
/tmp/ipykernel_1250/2997398431.py:138: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CFG.AMP):


fold 4 epoch 13/14: macro-F1 0.9520


/tmp/ipykernel_1250/2997398431.py:125: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CFG.AMP):
/tmp/ipykernel_1250/2997398431.py:138: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CFG.AMP):


fold 4 epoch 14/14: macro-F1 0.9546
--- Notebook 4 pooled OOF ---
official macro-F1: 0.9438 | accuracy: 0.9413
per-class F1: {'Idgham': 0.9361, 'Ikhfa': 0.9442, 'Iqlab': 0.9548, 'Izhar': 0.9398}
small-sheikh: 0.6046 | non-marsh: 0.6608 | marsh: 0.9811
per-fold macro-F1: [0.9476, 0.9263, 0.9533, 0.9321, 0.96]
training minutes: 10.0


In [13]:
# Test inference uses the exact best-epoch state paired with each fold's OOF predictions.
test_loader = make_loader(
    WaveDataset(WTE, np.arange(len(test_df)), labels=None, train=False),
    CFG.EVAL_BATCH,
    shuffle=False,
)
test_front = DeltaMelFront().to(DEVICE).eval()
TEST_PROBS = np.zeros((len(test_df), NCLS), dtype=np.float32)
inference_started = time.time()

for fold, state_dict in enumerate(best_states):
    model = build_pretrained_model(MODEL_SOURCE).to(
        DEVICE, memory_format=torch.channels_last
    )
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    fold_probabilities = []
    with torch.inference_mode():
        for waveforms in test_loader:
            waveforms = waveforms.to(DEVICE, non_blocking=True)
            features = test_front(waveforms, apply_time_warp=False).contiguous(memory_format=torch.channels_last)
            with torch.cuda.amp.autocast(enabled=CFG.AMP):
                logits = model(features)
            fold_probabilities.append(logits.float().softmax(1).cpu().numpy())
    TEST_PROBS += np.concatenate(fold_probabilities) / CFG.FOLDS
    print("inferred fold", fold)
    del model
    gc.collect()
    torch.cuda.empty_cache()

INFERENCE_SECONDS = time.time() - inference_started
assert TEST_PROBS.shape == (747, 4)
assert np.isfinite(TEST_PROBS).all()
assert np.allclose(TEST_PROBS.sum(axis=1), 1.0, atol=2e-5)
del test_front, test_loader
gc.collect()
torch.cuda.empty_cache()
print("test inference seconds:", round(INFERENCE_SECONDS, 1))


/tmp/ipykernel_1250/4180428360.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CFG.AMP):


inferred fold 0
inferred fold 1
inferred fold 2
inferred fold 3
inferred fold 4
test inference seconds: 12.0


In [14]:
# Save the exact shared comparison contract plus detailed diagnostics.
np.save(OUT / "oof_probs_notebook4.npy", OOF)
np.save(OUT / "test_probs_notebook4.npy", TEST_PROBS)

oof_frame = pd.DataFrame({
    "id": train_df.id.to_numpy(dtype=int),
    "fold": train_df.fold_v2.to_numpy(dtype=int),
    "label_name": train_df.label_name.to_numpy(),
    "sheikh_name": train_df.sheikh_name.to_numpy(),
    "true_class_index": labels.astype(int),
    "predicted_class": [CLASSES[index] for index in oof_predictions],
    "is_extreme_anomaly": train_df.is_extreme_anomaly.to_numpy(dtype=bool),
})
for class_index, class_name in enumerate(CLASSES):
    oof_frame[f"prob_{class_name}"] = OOF[:, class_index]
required_oof_columns = [
    "id", "fold", "label_name", "sheikh_name", "true_class_index",
    "predicted_class", *[f"prob_{name}" for name in CLASSES], "is_extreme_anomaly",
]
oof_frame = oof_frame[required_oof_columns]
oof_frame["group_id_v2"] = train_df.group_id_v2.to_numpy()
oof_frame["dur"] = train_df.dur.to_numpy()
oof_frame["is_train_representative"] = train_df.is_train_representative.to_numpy(dtype=bool)
oof_frame["center_shift_sec"] = train_df.center_shift_sec.to_numpy()
oof_frame["active_trim_start_sec"] = train_df.active_trim_start_sec.to_numpy()
oof_frame["active_trim_end_sec"] = train_df.active_trim_end_sec.to_numpy()
oof_frame["active_center_used"] = train_df.active_center_used.to_numpy(dtype=bool)
oof_frame["active_center_fallback"] = train_df.active_center_fallback.to_numpy(dtype=bool)
oof_frame["active_center_fallback_reason"] = train_df.active_center_fallback_reason.to_numpy()
oof_frame["oof_conf"] = OOF.max(axis=1)
oof_frame["oof_correct"] = oof_frame.predicted_class.eq(oof_frame.label_name)
assert oof_frame.id.tolist() == train_meta.id.tolist()
assert list(oof_frame.columns[:len(required_oof_columns)]) == required_oof_columns
oof_frame.to_csv(OUT / "oof_frame_notebook4.csv", index=False)

confidence_columns = [
    "id", "fold", "label_name", "predicted_class", "sheikh_name",
    "oof_conf", "is_extreme_anomaly", "dur", "center_shift_sec",
]
lowest_confidence = oof_frame.nsmallest(10, "oof_conf")[confidence_columns].copy()
lowest_confidence.insert(0, "confidence_group", "lowest")
highest_confidence = oof_frame.nlargest(10, "oof_conf")[confidence_columns].copy()
highest_confidence.insert(0, "confidence_group", "highest")
confidence_examples = pd.concat([lowest_confidence, highest_confidence], ignore_index=True)
confidence_examples.to_csv(OUT / "notebook4_confidence_examples.csv", index=False)

fold_metrics.to_csv(OUT / "notebook4_fold_metrics.csv", index=False)
FINAL_EVAL["per_sheikh"].to_csv(OUT / "notebook4_per_sheikh_metrics.csv", index=False)
confusion = confusion_matrix(labels, oof_predictions, labels=range(NCLS))
pd.DataFrame(confusion, index=CLASSES, columns=CLASSES).to_csv(
    OUT / "notebook4_confusion_matrix.csv", index_label="true_label"
)
class_report = classification_report(
    labels, oof_predictions, labels=range(NCLS), target_names=CLASSES,
    output_dict=True, zero_division=0,
)
pd.DataFrame(class_report).T.to_csv(OUT / "notebook4_classification_report.csv")

def subset_metrics(mask):
    mask = np.asarray(mask, dtype=bool)
    present = sorted(set(labels[mask].tolist()))
    return {
        "count": int(mask.sum()),
        "accuracy": float((labels[mask] == oof_predictions[mask]).mean()),
        "present_label_macro_f1": float(f1_score(
            labels[mask], oof_predictions[mask], average="macro",
            labels=present, zero_division=0,
        )),
    }

ROBUSTNESS_SUBSETS = {
    "long_clip_dur_gt_4s": subset_metrics(train_df.dur.to_numpy() > 4.0),
    "extreme_anomaly_flag": subset_metrics(train_df.is_extreme_anomaly.to_numpy()),
}
pd.DataFrame.from_dict(ROBUSTNESS_SUBSETS, orient="index").to_csv(
    OUT / "notebook4_subset_metrics.csv", index_label="subset"
)

center_diagnostics_frame = pd.concat([
    train_df[["id", *[
        "dur", "center_shift_sec", "active_trim_start_sec", "active_trim_end_sec",
        "active_center_used", "active_center_fallback", "active_center_fallback_reason",
    ]]].assign(split="train"),
    test_df[["id", *[
        "dur", "center_shift_sec", "active_trim_start_sec", "active_trim_end_sec",
        "active_center_used", "active_center_fallback", "active_center_fallback_reason",
    ]]].assign(split="test"),
], ignore_index=True)
center_diagnostics_frame = center_diagnostics_frame[["split", *center_diagnostics_frame.columns[:-1]]]
center_diagnostics_frame.to_csv(OUT / "notebook4_center_diagnostics.csv", index=False)

submission = sample_submission.copy()
submission["label_name"] = [CLASSES[index] for index in TEST_PROBS.argmax(axis=1)]
assert list(submission.columns) == list(sample_submission.columns) == ["id", "label_name"]
assert submission.id.tolist() == sample_submission.id.tolist() == test_df.id.tolist()
assert submission.id.is_unique and submission.label_name.notna().all()
assert set(submission.label_name) <= set(CLASSES)
submission.to_csv(OUT / "submission_notebook4.csv", index=False)

total_seconds = time.time() - T0
MODEL_CONFIG = {
    "notebook": 4,
    "experiment": "ConvNeXt-Tiny + log-mel/delta/delta-delta",
    "filename_note": "Requested destination filename retained; architecture is ConvNeXt V1 Tiny, not V2.",
    "competition": CFG.COMPETITION,
    "classes_probability_order": CLASSES,
    "primary_checkpoint": CFG.PRIMARY_CHECKPOINT,
    "fallback_checkpoint": CFG.FALLBACK_CHECKPOINT,
    "resolved_checkpoint": MODEL_SOURCE,
    "pretraining_compliance": "Generic supervised ImageNet-1K only; no audio, speech, Quran, Tajweed, CLIP/LAION, or ImageNet-22K weights.",
    "folds_source_sha256": FOLDS_SOURCE_SHA256,
    "fold_payload_sha256": FOLD_PAYLOAD_SHA256,
    "fold_payload_gzip_sha256": FOLD_PAYLOAD_GZIP_SHA256,
    "train_csv_sha256": EXPECTED_TRAIN_CSV_SHA256,
    "test_csv_sha256": EXPECTED_TEST_CSV_SHA256,
    "sample_rate": CFG.SR,
    "cache_seconds": CFG.CACHE_SEC,
    "crop_seconds": CFG.CROP_SEC,
    "active_center": {
        "eligible_clip_seconds_gt": CFG.ACTIVE_MIN_SEC,
        "rms_frame_ms": 40,
        "rms_hop_ms": 20,
        "smooth_frames": CFG.ACTIVE_SMOOTH_FRAMES,
        "minimum_q90_minus_q10_db": CFG.ACTIVE_MIN_DYNAMIC_DB,
        "threshold": "max(q10+6dB, q90-24dB)",
        "support": "at least 3 of 5 frames",
        "minimum_supported_activity_seconds": CFG.ACTIVE_MIN_SUPPORTED_SEC,
        "minimum_activity_span_seconds": CFG.ACTIVE_MIN_SPAN_SEC,
        "margin_seconds_each_side": CFG.ACTIVE_MARGIN_SEC,
        "maximum_center_shift_fraction": CFG.ACTIVE_MAX_SHIFT_FRAC,
        "fallback": "raw midpoint on nonfinite/uncertain/shift-too-large; raw midpoint plus symmetric padding for short clips",
        "deletes_clips": False,
        "diagnostics": CENTER_DIAGNOSTICS,
    },
    "augmentation_order": [
        "waveform center jitter and gain (training only)",
        "raw log-mel dB",
        "time-axis spectrogram warp (training only)",
        "delta then delta-delta",
        "independent per-channel global standardization",
        "shared-channel SpecAugment (training only)",
    ],
    "train_center_jitter_seconds": CFG.JITTER_SEC,
    "train_gain_db": [-CFG.GAIN_DB, CFG.GAIN_DB],
    "frontend": {
        "channels": ["raw_logmel_db", "delta_time_win9", "delta_delta_time_win9"],
        "shape": [3, 128, 401],
        "n_fft": CFG.N_FFT,
        "hop_length": CFG.HOP,
        "n_mels": CFG.N_MELS,
        "f_min": CFG.FMIN,
        "f_max": CFG.FMAX,
        "delta_win_length": CFG.DELTA_WIN,
        "delta_padding": "replicate",
        "normalization": "independent global standardization per channel",
        "time_warp": {
            "probability": CFG.TIME_WARP_PROB,
            "rate": [CFG.TIME_WARP_RATE_MIN, CFG.TIME_WARP_RATE_MAX],
            "axis": "time only on raw log-mel dB",
            "restore_length": "center crop or replicate pad",
            "before_deltas": True,
            "validation_test_enabled": False,
        },
        "specaugment": {
            "frequency_masks": 2,
            "frequency_max_width": CFG.FREQ_MASK_MAX,
            "time_masks": 2,
            "time_max_width": CFG.TIME_MASK_MAX,
            "probability_each": 0.5,
            "geometry_shared_across_channels": True,
        },
    },
    "training": {
        "folds": CFG.FOLDS,
        "epochs": CFG.EPOCHS,
        "batch_size": CFG.BATCH,
        "optimizer": "AdamW",
        "learning_rate": CFG.LR,
        "weight_decay": CFG.WD,
        "scheduler": "OneCycleLR",
        "label_smoothing": CFG.LABEL_SMOOTH,
        "dropout": CFG.DROPOUT,
        "drop_path": CFG.DROP_PATH,
        "amp": CFG.AMP,
        "channels_last": True,
        "seed": SEED,
        "exact_dedup_train_only": True,
        "balanced_sampling": False,
        "cmvn": False,
        "anomaly_removal": False,
        "exact_match_patch": False,
        "tta": False,
    },
    "expected_t4_runtime_minutes": CFG.EXPECTED_RUNTIME_MIN,
}

METRICS_REPORT = {
    "official_metric": "f1_score(y_true, y_pred, average='macro')",
    "pooled_macro_f1": FINAL_EVAL["macro_f1"],
    "accuracy": FINAL_EVAL["accuracy"],
    "per_fold_macro_f1": recomputed_fold_f1,
    "fold_macro_f1_mean": float(np.mean(recomputed_fold_f1)),
    "fold_macro_f1_std": float(np.std(recomputed_fold_f1)),
    "per_class_f1": FINAL_EVAL["per_class"],
    "small_sheikh_macro_f1": FINAL_EVAL["small_sheikh_macro_f1"],
    "marsh_macro_f1": FINAL_EVAL["marsh_macro_f1"],
    "non_marsh_macro_f1": FINAL_EVAL["non_marsh_macro_f1"],
    "mean_sheikh_accuracy": FINAL_EVAL["mean_sheikh_acc"],
    "mean_present_label_macro_f1_per_sheikh": FINAL_EVAL["mean_present_macro_f1"],
    "per_sheikh": FINAL_EVAL["per_sheikh"].to_dict(orient="records"),
    "robustness_subsets": ROBUSTNESS_SUBSETS,
    "active_center_diagnostics": CENTER_DIAGNOSTICS,
    "training_seconds": TRAIN_SECONDS,
    "inference_seconds": INFERENCE_SECONDS,
    "total_runtime_seconds": total_seconds,
    "test_prediction_counts": submission.label_name.value_counts().reindex(CLASSES, fill_value=0).to_dict(),
}

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Not JSON serializable: {type(value)}")

with (OUT / "CONFIG_notebook4.json").open("w", encoding="utf-8") as handle:
    json.dump(MODEL_CONFIG, handle, indent=2, ensure_ascii=False, default=json_default)
with (OUT / "notebook4_metrics_report.json").open("w", encoding="utf-8") as handle:
    json.dump(METRICS_REPORT, handle, indent=2, ensure_ascii=False, default=json_default)

def frame_to_markdown(frame):
    try:
        return frame.to_markdown(index=False)
    except Exception:
        return "```\n" + frame.to_string(index=False) + "\n```"

report_lines = [
    "# Notebook 4 - ConvNeXt-Tiny + delta channels",
    "",
    f"- Resolved checkpoint: `{MODEL_SOURCE}`",
    f"- Pooled OOF macro-F1: {FINAL_EVAL['macro_f1']:.6f}",
    f"- Per-fold macro-F1: {[round(x, 6) for x in recomputed_fold_f1]}",
    f"- Per-class F1: { {k: round(v, 6) for k, v in FINAL_EVAL['per_class'].items()} }",
    f"- Small-sheikh stress: {FINAL_EVAL['small_sheikh_macro_f1']:.6f}",
    f"- Non-marsh stress: {FINAL_EVAL['non_marsh_macro_f1']:.6f}",
    f"- Long clips (>4 s): {ROBUSTNESS_SUBSETS['long_clip_dur_gt_4s']}",
    f"- Extreme anomaly flag: {ROBUSTNESS_SUBSETS['extreme_anomaly_flag']}",
    f"- Active-center train diagnostics: {CENTER_DIAGNOSTICS['train']}",
    f"- Active-center test diagnostics: {CENTER_DIAGNOSTICS['test']}",
    f"- Training minutes: {TRAIN_SECONDS / 60:.2f}",
    f"- Inference seconds: {INFERENCE_SECONDS:.1f}",
    "",
    "## Exact augmentation order",
    "",
    "Training: active-center cache -> 4 s crop with +/-0.20 s jitter and +/-6 dB gain -> raw log-mel dB -> time-only warp rate 0.96-1.04 (p=0.30) -> delta -> delta-delta -> per-channel standardization -> shared masks (2 frequency max12, 2 time max28, p=0.5 each).",
    "",
    "Validation/test: deterministic active-center 4 s crop -> raw log-mel dB -> delta -> delta-delta -> per-channel standardization. No jitter, gain, warp, SpecAugment, patch, or TTA.",
    "",
    "## Lowest-confidence OOF examples",
    "",
    frame_to_markdown(lowest_confidence),
    "",
    "## Highest-confidence OOF examples",
    "",
    frame_to_markdown(highest_confidence),
    "",
    "Selection used frozen CV only; no public leaderboard feedback.",
    "Probability column order: Idgham, Ikhfa, Iqlab, Izhar.",
]
(OUT / "REPORT_notebook4.md").write_text("\n".join(report_lines) + "\n", encoding="utf-8")

artifact_names = [
    "oof_probs_notebook4.npy",
    "test_probs_notebook4.npy",
    "oof_frame_notebook4.csv",
    "CONFIG_notebook4.json",
    "REPORT_notebook4.md",
    "submission_notebook4.csv",
    "notebook4_fold_metrics.csv",
    "notebook4_per_sheikh_metrics.csv",
    "notebook4_confusion_matrix.csv",
    "notebook4_classification_report.csv",
    "notebook4_metrics_report.json",
    "notebook4_subset_metrics.csv",
    "notebook4_center_diagnostics.csv",
    "notebook4_confidence_examples.csv",
]
manifest = {name: sha256_file(OUT / name) for name in artifact_names}
with (OUT / "notebook4_artifact_sha256.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

zip_path = Path("/content/notebook4_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.iterdir()):
        archive.write(path, arcname=path.name)

print("saved", len(list(OUT.iterdir())), "artifacts to", OUT)
print("download bundle:", zip_path)
print("submission/sample schema and ID assertions: PASSED")
print("lowest-confidence examples:\n", lowest_confidence.to_string(index=False))
print("highest-confidence examples:\n", highest_confidence.to_string(index=False))
print("OOF pooled macro-F1:", round(FINAL_EVAL["macro_f1"], 6))


saved 15 artifacts to /content/notebook4_outputs
download bundle: /content/notebook4_outputs.zip
submission/sample schema and ID assertions: PASSED
lowest-confidence examples:
 confidence_group   id  fold label_name predicted_class      sheikh_name  oof_conf  is_extreme_anomaly   dur  center_shift_sec
          lowest 1321     1      Ikhfa          Idgham abdulkadr othman  0.313336               False 0.256               0.0
          lowest    3     0      Iqlab           Iqlab    abdulah_marsh  0.325602                True 0.256               0.0
          lowest  320     0      Iqlab           Iqlab    abdulah_marsh  0.331177                True 0.256               0.0
          lowest 1167     2      Ikhfa           Ikhfa    abdulah_marsh  0.338413               False 3.299               0.0
          lowest  281     0     Idgham           Ikhfa  rasheed alrahbi  0.341526               False 0.438               0.0
          lowest 1499     3      Ikhfa           Ikhfa    abdulah_m

## Run and handoff

Use **Runtime -> Run all** on one Colab T4 after placing `kaggle.json` at `/content/kaggle.json`. The final cell creates `/content/notebook4_outputs.zip` plus the unzipped artifacts in `/content/notebook4_outputs/`.

Required shared files are `submission_notebook4.csv`, `oof_probs_notebook4.npy`, `test_probs_notebook4.npy`, `oof_frame_notebook4.csv`, `CONFIG_notebook4.json`, and `REPORT_notebook4.md`. The report includes long-clip/anomaly metrics, active-center diagnostics, and the lowest/highest-confidence OOF examples.

Compare this model with Notebooks 3 and 5 only through the shared frozen folds: pooled/per-fold macro-F1, per-class F1, sheikh and subset stress metrics, and OOF probability/error correlation. Do not choose it from the 8% public leaderboard.
